In [1]:
import requests
import os
import time
import pandas as pd
import urllib.parse

species_list = [
    "Trichoderma reesei",
    "Trichoderma virens",
    "Trichoderma atroviride",
    "Trichoderma harzianum",
    "Aspergillus niger",
    "Aspergillus oryzae",
    "Aspergillus nidulans",
    "Aspergillus brasiliensis",
    "Aspergillus japonicus",
    "Aspergillus fumigatus",
    "Aspergillus terreus",
    "Aspergillus clavatus",
    "Penicillium chrysogenum",
    "Penicillium digitatum",
    "Penicillium oxalicum",
    "Penicillium roqueforti",
    "Penicillium echinulatum",
    "Neurospora crassa",
    "Fusarium graminearum",
    "Fusarium oxysporum",
    "Pleurotus ostreatus",
    "Talaromyces pinophilus",
    "Talaromyces amestolkiae",
    "Yarrowia lipolytica",
    "Rhizopus oryzae",
    "Chaetomium globosum",
    "Podospora anserina",
    "Chaetomium thermophilum",
    "Myceliophthora thermophila",
    "Thielavia terrestris"
]


In [1]:
import requests
import os
import time
import pandas as pd
import urllib.parse

species_list = ["Penicillium funiculosum", "pseudomonas putida"]


In [2]:
output_dir = "../data/proteomes"
os.makedirs(output_dir, exist_ok=True)

report = []


def count_fasta_entries(fasta_text):
    return fasta_text.count("\n>") + (1 if fasta_text.startswith(">") else 0)


def download_full_proteome(species, output_dir):

    print(f"Fallback: searching taxonomy for {species}")

    tax_url = (
        "https://rest.uniprot.org/taxonomy/search"
        f"?query={urllib.parse.quote(species)}&format=json&size=10"
    )

    response = requests.get(tax_url)

    if response.status_code != 200:
        print("Taxonomy search failed")
        return None, None, None, "FALLBACK_FAILED"

    results = response.json().get("results", [])

    candidates = []

    for r in results:

        tax_id = r.get("taxonId")
        name = r.get("scientificName", "")

        if not tax_id or "virus" in name.lower():
            continue

        query = f"organism_id:{tax_id}"
        encoded_query = urllib.parse.quote(query)

        test_url = (
            "https://rest.uniprot.org/uniprotkb/stream"
            f"?format=fasta&query={encoded_query}"
        )

        test_response = requests.get(test_url)

        if test_response.status_code == 200 and len(test_response.text) > 0:

            fasta_text = test_response.text
            protein_count = count_fasta_entries(fasta_text)

            # keep only meaningful proteomes
            if protein_count > 3000:

                candidates.append({
                    "tax_id": tax_id,
                    "organism": name,
                    "protein_count": protein_count,
                    "fasta": fasta_text
                })

    if not candidates:
        print("No valid fallback proteome found")
        return None, None, None, "FALLBACK_FAILED"

    best = sorted(
        candidates,
        key=lambda x: x["protein_count"],
        reverse=True
    )[0]

    print(
        f"Selected fallback tax_id: "
        f"{best['tax_id']} "
        f"({best['protein_count']} proteins)"
    )

    filename = species.lower().replace(" ", "_") + ".fasta"

    filepath = os.path.join(
        output_dir,
        filename
    )

    with open(filepath, "w") as f:
        f.write(best["fasta"])

    return (
        filename,
        best["tax_id"],
        best["protein_count"],
        "FALLBACK_OK"
    )


# =====================================
# MAIN LOOP
# =====================================

for species in species_list:

    print(f"\nSearching: {species}")

    search_url = (
        "https://rest.uniprot.org/proteomes/search"
        f"?query=organism_name:\"{species}\" AND reference:true"
        "&format=json&size=10"
    )

    response = requests.get(search_url)

    if response.status_code != 200:

        print("Search failed")

        report.append([
            species,
            "NA",
            "NA",
            "NA",
            "NA",
            "NA",
            "FAILED_SEARCH"
        ])

        continue

    results = response.json().get("results", [])

    candidates = []

    for r in results:

        proteome_id = r.get("id")

        organism = (
            r.get("taxonomy", {})
            .get("scientificName", "")
        )

        tax_id = (
            r.get("taxonomy", {})
            .get("taxonId", "")
        )

        size = r.get("proteinCount", 0)

        if not proteome_id:
            continue

        if "virus" in organism.lower():
            continue

        candidates.append({
            "proteome_id": proteome_id,
            "tax_id": tax_id,
            "organism": organism,
            "size": size
        })

    # =====================================
    # FALLBACK
    # =====================================

    if not candidates:

        print(
            "No reference proteome found → using fallback"
        )

        (
            filename,
            tax_id,
            protein_count,
            status
        ) = download_full_proteome(
            species,
            output_dir
        )

        report.append([
            species,
            "NA",
            tax_id if tax_id else "NA",
            "NA",
            protein_count if protein_count else "NA",
            filename if filename else "NA",
            status
        ])

        continue

    # =====================================
    # SELECT BEST REFERENCE PROTEOME
    # =====================================

    filtered = [
        c for c in candidates
        if 8000 <= c["size"] <= 20000
    ]

    if filtered:
        best = sorted(
            filtered,
            key=lambda x: x["size"],
            reverse=True
        )[0]

    else:
        best = sorted(
            candidates,
            key=lambda x: x["size"],
            reverse=True
        )[0]

    proteome_id = best["proteome_id"]

    print(
        f"Selected: {proteome_id} "
        f"({best['size']} proteins)"
    )

    fasta_url = (
        "https://rest.uniprot.org/uniprotkb/stream"
        f"?format=fasta&query=(proteome:{proteome_id})"
    )

    fasta_response = requests.get(fasta_url)

    if (
        fasta_response.status_code == 200
        and len(fasta_response.text) > 0
    ):

        fasta_text = fasta_response.text

        protein_count = count_fasta_entries(
            fasta_text
        )

        filename = (
            species.lower()
            .replace(" ", "_")
            + ".fasta"
        )

        filepath = os.path.join(
            output_dir,
            filename
        )

        with open(filepath, "w") as f:
            f.write(fasta_text)

        report.append([
            species,
            proteome_id,
            best["tax_id"],
            best["organism"],
            protein_count,
            filename,
            "OK"
        ])

        print(
            f"Downloaded: {filename}"
        )

    else:

        report.append([
            species,
            proteome_id,
            best["tax_id"],
            "NA",
            "NA",
            "NA",
            "FAILED_DOWNLOAD"
        ])

        print("Download failed")

    time.sleep(1)


# =====================================
# SAVE REPORT
# =====================================

df = pd.DataFrame(
    report,
    columns=[
        "species_query",
        "proteome_id",
        "tax_id",
        "selected_organism",
        "protein_count",
        "file",
        "status"
    ]
)

df.to_csv(
    "../data/proteomes/reference_proteomes_report.csv",
    index=False
)

print("Done")


Searching: Penicillium funiculosum
No reference proteome found → using fallback
Fallback: searching taxonomy for Penicillium funiculosum
Selected fallback tax_id: 128442 (11075 proteins)

Searching: pseudomonas putida
Selected: UP000000556 (5527 proteins)
Downloaded: pseudomonas_putida.fasta
Done
